In [1]:
%load_ext autoreload
%autoreload 2

## Data curation

Curating raw data with the enhanced chat client

In [ ]:
import os

import dotenv

dotenv.load_dotenv()

DATA_DIR = os.getenv("DATA_DIR")

In [ ]:
# load the data we need
import json

import pandas as pd

drug_df = pd.read_csv(os.path.join(DATA_DIR, "rxrx_groundtruth_compounds.csv"))
display(drug_df.head())

# load the action primitives
with open(os.path.join(DATA_DIR, "action_primitives.json")) as f:
    action_primitives = json.load(f)

action_primitives_list = [x["action"] for x in action_primitives]
print(action_primitives_list)

,drug name,target,type,iupac_name_pubchem,smiles_pubchem,InChI,cdd_created_at,updated_at,cas_number,cns_mpo_score,...,smiles_rxrx,topological_polar_surface_area,udfs,stereochemistry,scaffold,fsp3,log_d,targets,id,datastream_metadata
0,A8301,ALK,inhibitor,3-(6-methylpyridin-2-yl)-N-phenyl-4-quinolin-4...,CC1=NC(=CC=C1)C2=NN(C=C2C3=CC=NC4=CC=CC=C34)C(...,InChI=1S/C25H19N5S/c1-17-8-7-13-23(27-17)24-21...,2018-08-15 19:45:02+00:00,2025-02-25 23:24:13.714366+00:00,NaN,3.39384,...,CC1=CC=CC(C2=NN(C(=S)NC3=CC=CC=C3)C=C2C2=CC=NC...,55.63,NaN,NaN,NaN,0.040000,5.10041,[],327020.0,{'uuid': '13cab22b-2a29-4136-ae0d-524010100110...
1,ACHP,IKK,inhibitor,2-amino-6-[2-(cyclopropylmethoxy)-6-hydroxyphe...,C1CC1COC2=CC=CC(=C2C3=NC(=C(C(=C3)C4CCNCC4)C#N...,InChI=1S/C21H24N4O2/c22-11-16-15(14-6-8-24-9-7...,2017-07-04 17:23:17+00:00,2025-02-25 23:42:59.582801+00:00,NaN,3.52522,...,NC1=NC(C2=C(O)C=CC=C2OCC2CC2)=CC(C2CCNCC2)=C1C#N,104.19,NaN,NaN,NaN,0.428571,1.53928,[],1560419.0,{'uuid': '91ab83f1-8674-419b-8b5e-6ffb01010111...
2,AP26113,ALK,inhibitor,5-chloro-2-N-[4-[4-(dimethylamino)piperidin-1-...,CN(C)C1CCN(CC1)C2=CC(=C(C=C2)NC3=NC=C(C(=N3)NC...,InChI=1S/C26H34ClN6O2P/c1-32(2)18-12-14-33(15-...,2017-04-06 16:09:21+00:00,2025-07-10 22:57:25.278245+00:00,1197958-12-5,3.01000,...,COC1=CC(N2CCC(N(C)C)CC2)=CC=C1NC1=NC=C(Cl)C(NC...,82.62,NaN,NaN,NaN,0.384615,1.83076,"['ALK,EGFR']",1612394.0,{'uuid': 'd813f8a5-9f16-4977-943b-7bb411101100...
3,ASP3026,ALK,inhibitor,2-N-[2-methoxy-4-[4-(4-methylpiperazin-1-yl)pi...,CC(C)S(=O)(=O)C1=CC=CC=C1NC2=NC(=NC=N2)NC3=C(C...,"InChI=1S/C29H40N8O3S/c1-21(2)41(38,39)27-8-6-5...",2017-07-04 17:23:18+00:00,2025-03-20 21:14:10.380040+00:00,NaN,3.03352,...,COC1=C(NC2=NC=NC(NC3=C(S(=O)(=O)C(C)C)C=CC=C3)...,115.82,NaN,NaN,NaN,0.482759,1.90482,['ALK'],113310.0,{'uuid': '146053cb-cfa4-4e1a-b220-86e800100101...
4,AZ 960,JAK,inhibitor,5-fluoro-2-[[(1S)-1-(4-fluorophenyl)ethyl]amin...,CC1=CC(=NN1)NC2=C(C=C(C(=N2)N[C@@H](C)C3=CC=C(...,InChI=1S/C18H16F2N6/c1-10-7-16(26-25-10)23-18-...,2017-04-06 17:55:00+00:00,2024-08-21 15:53:17.597471+00:00,NaN,3.65631,...,C[C@H](NC1=NC(NC2=NNC(C)=C2)=C(F)C=C1C#N)C1=CC...,89.42,NaN,NaN,NaN,0.166667,3.64288,['JAK'],1736727.0,{'uuid': 'af8760f5-7b3f-4a5d-a3f7-59f300000000...


['binds_to', 'modulates_activity', 'regulates_expression', 'modulates_complex', 'causes_phenotype', 'rescues_phenotype', 'similar_to', 'correlates_with', 'participates_in', 'gain_of_function', 'loss_of_function', 'regulates_translation', 'post_translational_modification', 'localises_to', 'converts_substrate', 'degrades_or_stabilises', 'cell_cell_interaction', 'chromatin_modification', 'set_context']


In [5]:
import fsspec
import yaml

with fsspec.open(os.path.join(DATA_DIR, "GroundTruthPacketV2032720_summary.yaml"))as f:
    ground_truth = yaml.safe_load(f)


def build_inputs(ground_truth):
    input_data = []
    for disease in ground_truth:
        context = {}

        context_perturbation = ground_truth[disease]["perturbation"]
        context["perturbation_type"] = context_perturbation.get("type", "N/A")
        context["description"] = context_perturbation.get("name", "N/A")
        context["cell_type"] = "N/A"
        context["disease_model"] = disease

        for drug_data in ground_truth[disease]["drug"]:
            for drug_name, drug_info in drug_data.items():
                drug_perturbation = {}
                drug_perturbation["type"] = "chemical"

                # Safely get the first matching SMILES or None if no match found
                matching_smiles = drug_df[drug_df["drug name"] == drug_name]["smiles_pubchem"]
                drug_perturbation["smiles"] = matching_smiles.iloc[0] if not matching_smiles.empty else None
                drug_perturbation["name"] = drug_name
                drug_perturbation["target"] = drug_info["target"]
                drug_perturbation["moa_type"] = drug_info["type"]
                input_data.append({"context": context, "perturbation": drug_perturbation})
    return input_data

In [6]:
pert_path = os.path.join(DATA_DIR, "perturbations.json")
if not os.path.exists(pert_path):
    perturbation_cell_context = build_inputs(ground_truth)
else:
    with open(pert_path) as f:
        perturbation_cell_context = json.load(f)

### Load enhanced chat

In [3]:
from explain.enhanced_chat._access_token import get_access_token

access_token = get_access_token(auto_open=True)

2025-07-30 13:54:31.419 | INFO     | explain.enhanced_chat._access_token:get_access_token:27 - Starting OAuth device code flow
2025-07-30 13:54:32.043 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:44 - ============================================================
2025-07-30 13:54:32.043 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:45 - AUTHENTICATION REQUIRED
2025-07-30 13:54:32.044 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:46 - ============================================================
2025-07-30 13:54:32.044 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:47 - Please open the following URL in your browser:
2025-07-30 13:54:32.044 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:48 - 
https://okta.recursionpharma.com/activate?user_code=LNNWTQVQ

2025-07-30 13:54:32.044 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:49 - Or go to: https://okta.recursionpharma.com/acti

.

2025-07-30 13:54:37.536 | DEBUG    | explain.enhanced_chat._access_token:get_access_token:72 - Polling attempt 2
2025-07-30 13:54:38.097 | SUCCESS  | explain.enhanced_chat._access_token:get_access_token:95 - Authentication successful!


In [7]:
from enhanced_chat_client import AuthenticatedClient

client = AuthenticatedClient(base_url="https://enhanced-chat.centaur-platform-dev.com/", token=access_token)
client = await client.__aenter__()

Build the enhanced chat client for launching jobs

In [8]:
from enhanced_chat_client.models.model import Model
from enhanced_chat_client.models.output_style import OutputStyle

from explain.enhanced_chat.manager import EnhancedChatManager

chat_manager = EnhancedChatManager(client)
chat_manager.update_config(
    model=Model.CLAUDE_V3_7_SONNET,
    kg_rag_enabled=True,
    omim_enabled=True,
    pubmed_enabled=True,
    fulltext_enabled=True,
    citeline_enabled=True,
    kg_agent_enabled=True,
    max_pubmed_to_read=50,
    max_fulltext_to_read=20,
    max_pdf_index_to_read=20,
    output_style=OutputStyle.NATURE,
    is_shareable=True,
)

In [9]:
from datetime import datetime

report_template = open(os.path.join(DATA_DIR, "templates/generate-report.txt")).read()
job_tag = "hooke-explain-report-{}".format(datetime.now().strftime("%Y%m%d%H%M"))

Use a help script to run the job. Here only demonstrating it on a few samples for efficiency

In [11]:
from curate_utils import process_treatments_efficiently

# Now run the efficient processing
treatments = perturbation_cell_context[:5]
chunk_size = 1

print(f"🏁 Starting efficient parallel processing with base job tag: {job_tag}")

# Run the efficient processing
processing_results = await process_treatments_efficiently(
    treatments=treatments,
    chat_manager=chat_manager,
    report_template=report_template,
    base_job_tag=job_tag,
    chunk_size=chunk_size,
)

# Extract the outputs in the original format for compatibility
outputs = []
for result in processing_results["all_results"]:
    outputs.append(
        {
            "prompts": result.get("prompts"),
            "output": result["output"],
            "chunk": result["chunk_index"],
            "job_tag": result["job_tag"],
            "status": result["status"],
        }
    )

🏁 Starting efficient parallel processing with base job tag: hooke-explain-report-202507301355
📊 Processing 5 treatments in 5 parallel chunks...
🚀 Launching chunk 2 with 1 treatments...
🚀 Launching chunk 0 with 1 treatments...
🚀 Launching chunk 4 with 1 treatments...
🚀 Launching chunk 1 with 1 treatments...
🚀 Launching chunk 3 with 1 treatments...


2025-07-30 13:56:14.860 | INFO     | explain.enhanced_chat.manager:submit_and_collect:382 - Job hooke-explain-report-202507301355-chunk-2 submitted
2025-07-30 13:56:14.868 | INFO     | explain.enhanced_chat.manager:submit_and_collect:382 - Job hooke-explain-report-202507301355-chunk-4 submitted
2025-07-30 13:56:14.869 | INFO     | explain.enhanced_chat.manager:submit_and_collect:382 - Job hooke-explain-report-202507301355-chunk-0 submitted
2025-07-30 13:56:14.936 | INFO     | explain.enhanced_chat.manager:submit_and_collect:382 - Job hooke-explain-report-202507301355-chunk-1 submitted
2025-07-30 13:56:15.007 | INFO     | explain.enhanced_chat.manager:submit_and_collect:382 - Job hooke-explain-report-202507301355-chunk-3 submitted
Job hooke-explain-report-202507301355-chunk-1:   0%|          | 0/1 [00:00<?, ?tasks/s]











Job hooke-explain-report-202507301355-chunk-1:   0%|          | 0/1 [00:03<?, ?tasks/s]





Job hooke-explain-report-202507301355-chunk-1:   0%|          | 0/1 


📈 Processing Summary:
   ⏱️  Total time: 424.0 seconds
   ✅ Successful: 5
   🔄 Reused existing: 0
   ❌ Failed: 0
   📊 Total chunks: 5


In [31]:
# Utility functions for monitoring and management


async def check_all_jobs_status(base_job_tag, num_chunks, chunk_size=1):
    """Check status of all chunk jobs"""
    statuses = []
    for i in range(0, num_chunks * chunk_size, chunk_size):  # Assuming chunk_size=20
        job_tag = f"{base_job_tag}-chunk-{i}"
        try:
            status = await chat_manager.check_job_status(job_tag)
            statuses.append({"chunk": i, "job_tag": job_tag, "status": status})
        except Exception as e:
            statuses.append({"chunk": i, "job_tag": job_tag, "error": str(e)})
    return statuses


async def collect_existing_results(base_job_tag, num_chunks, chunk_size=1):
    """Collect results from existing completed jobs"""
    results = []
    for i in range(0, num_chunks * chunk_size, chunk_size):  # Assuming chunk_size=20
        job_tag = f"{base_job_tag}-chunk-{i}"
        try:
            status = await chat_manager.check_job_status(job_tag)
            if status.get("is_complete", False):
                output = await chat_manager.collect_results(job_tag=job_tag, wait_for_completion=False, refetch=True)
                results.append({"chunk_index": i, "job_tag": job_tag, "output": output, "status": "collected"})
                print(f"✓ Collected results for chunk {i}")
        except Exception as e:
            print(f"❌ Error collecting chunk {i}: {e}")
    return results

In [32]:
statuses = await check_all_jobs_status(job_tag, len(treatments))
results = await collect_existing_results(job_tag, len(treatments))

2025-07-30 14:12:25.696 | INFO     | explain.enhanced_chat.manager:collect_results:282 - Job hooke-explain-report-202507301355-chunk-0 found completed on server but not tracked locally


✓ Collected results for chunk 0


2025-07-30 14:12:26.486 | INFO     | explain.enhanced_chat.manager:collect_results:282 - Job hooke-explain-report-202507301355-chunk-1 found completed on server but not tracked locally


✓ Collected results for chunk 1


2025-07-30 14:12:27.519 | INFO     | explain.enhanced_chat.manager:collect_results:282 - Job hooke-explain-report-202507301355-chunk-2 found completed on server but not tracked locally


✓ Collected results for chunk 2


2025-07-30 14:12:28.326 | INFO     | explain.enhanced_chat.manager:collect_results:282 - Job hooke-explain-report-202507301355-chunk-3 found completed on server but not tracked locally


✓ Collected results for chunk 3


2025-07-30 14:12:29.470 | INFO     | explain.enhanced_chat.manager:collect_results:282 - Job hooke-explain-report-202507301355-chunk-4 found completed on server but not tracked locally


✓ Collected results for chunk 4


Let's reparse the question, template and results into a dataframe to be used later

In [33]:
from explain.data.curation.prompt_parser import PromptParser

parser = PromptParser()

report_results = []
for res in results:
    for conv in res["output"].conversations:
        datum = parser(conv.messages[0].to_dict()["content"])
        datum["report_text"] = conv.messages[-1].to_dict()["content"]
        datum["report"] = chat_manager.extract_messages([conv])[-1]["messages"][-1]
        report_results.append(datum)

df = pd.DataFrame(report_results)

In [34]:
df

,perturbation,question,report_text,report
0,{'context': {'perturbation_type': 'soluble fac...,How does the following perturbation influence ...,\n# Mechanistic Report: Bevacizumab-Mediated I...,{'content': ' # Mechanistic Report: Bevacizuma...
1,{'context': {'perturbation_type': 'soluble fac...,How does the following perturbation influence ...,\n# Nintedanib as a Multi-Targeted Angiokinase...,{'content': ' # Nintedanib as a Multi-Targeted...
2,{'context': {'perturbation_type': 'soluble fac...,How does the following perturbation influence ...,\n# Mechanistic Report: Apatinib in VEGF-Drive...,{'content': ' # Mechanistic Report: Apatinib i...
3,{'context': {'perturbation_type': 'soluble fac...,How does the following perturbation influence ...,\n# Mechanistic Report: Lenvatinib's Effects o...,{'content': ' # Mechanistic Report: Lenvatinib...
4,{'context': {'perturbation_type': 'soluble fac...,How does the following perturbation influence ...,\n# Mechanistic Report: Motesanib as a VEGF Pa...,{'content': ' # Mechanistic Report: Motesanib ...


### Translate report into structured explanation

Uinsg Claude 4 (or any other LLM), reformulate the report into the final result

In [37]:
from explain.data.curation.struct_explainer import create_structure_explainer

# just use the enhanced strategy
data = df.to_dict(orient="records")
# Generate enhanced explanation
async with create_structure_explainer(report_processing_strategy="enhanced") as explainer:
    result = await explainer.process_from_dataframe(
        df,
        show_progress=True,
        max_concurrent=10,
    )

2025-07-30 14:34:12.618 | INFO     | explain.llm._client:__init__:44 - Initialized Anthropic Vertex clients for us-east5
2025-07-30 14:34:12.620 | INFO     | explain.data.curation.struct_explainer:process_batch:136 - Processing 5 items with max_concurrent=10
Processing explanations: 100%|██████████| 5/5 [00:26<00:00,  5.33s/it]


In [38]:
result.head()

,index,question,thinking,answer,explain,dag,raw_response,success,error,input_perturbation,input_report_text
0,0,How does the following perturbation influence ...,Let me work through this step by step.\n\nThe ...,Bevacizumab is a humanized monoclonal antibody...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nLet me work through this step by step...,True,None,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Bevacizumab-Mediated I...
1,1,How does the following perturbation influence ...,Let me analyze this step by step:\n\n1. **Cont...,In the context of VEGF-stimulated tumor angiog...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n3""...",<think>\nLet me analyze this step by step:\n\n...,True,None,{'context': {'perturbation_type': 'soluble fac...,\n# Nintedanib as a Multi-Targeted Angiokinase...
2,2,How does the following perturbation influence ...,Let me analyze this step by step:\n\n1. Contex...,Apatinib binds to the ATP-binding site of VEGF...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nLet me analyze this step by step:\n\n...,True,None,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Apatinib in VEGF-Drive...
3,3,How does the following perturbation influence ...,Let me analyze this step by step:\n\n1. Contex...,Lenvatinib binds to the ATP-binding pocket of ...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n3"", relation=""causal"")\nedge(""n2""...",<think>\nLet me analyze this step by step:\n\n...,True,None,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Lenvatinib's Effects o...
4,4,How does the following perturbation influence ...,Let me analyze this step by step:\n\n1. **Cont...,Motesanib acts as a potent ATP-competitive inh...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n4"", relation=""causal"")\nedge(""n2""...",<think>\nLet me analyze this step by step:\n\n...,True,None,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Motesanib as a VEGF Pa...
